# ARS 번호로 STCIS 정류장 ID 역조회

`sttnArsno`를 국토부 STCIS `bussttn` API로 조회해 `sttnId`와 정류장명을 가져오고, 위경도 통합 파일의 누락 정류장명을 보정합니다.

ARS 입력파일(`route_stop_ars_reference.csv`)은 `노선번호,sttnArsno,sdCd,sggCd` 컬럼으로 준비합니다.

### 셀 1. 2200번 누락 구간 ARS 정류장 조회\n

### ?? ? 1. ??? ?????BusStop API ?? ??


In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
ENDPOINT = 'https://stcis.go.kr/openapi/bussttn.json'
API_KEY = getpass('STCIS 인증키를 입력하세요: ').strip()
DATE = '20241017'
ARS_INPUT = DATA_DIR / 'route_stop_ars_reference.csv'
if not API_KEY: raise ValueError('인증키가 입력되지 않았습니다.')

# 예시 형식: 실제 ARS 목록을 이 CSV에 입력
# 노선번호,sttnArsno,sdCd,sggCd
# 1100,36048,41,41287
# 1200,36118,41,41285
# 2200,30091,41,41480
if not ARS_INPUT.exists():
    raise FileNotFoundError(f'ARS 입력파일이 없습니다: {ARS_INPUT}\n노선번호,sttnArsno,sdCd,sggCd 형식으로 만들어 주세요.')
ars = pd.read_csv(ARS_INPUT, dtype=str, encoding='utf-8-sig').fillna('')
required = {'노선번호', 'sttnArsno', 'sdCd'}
if not required.issubset(ars.columns): raise ValueError(f'필수 컬럼 누락: {required - set(ars.columns)}')
ars['sttnArsno'] = ars['sttnArsno'].str.strip().str.zfill(5)
ars['sdCd'] = ars['sdCd'].str.strip()
if 'sggCd' not in ars.columns: ars['sggCd'] = ''
targets = ars.drop_duplicates(['sttnArsno', 'sdCd', 'sggCd']).reset_index(drop=True)

def parse(payload):
    result = payload.get('result', [])
    if isinstance(result, dict): result = [result]
    return result if isinstance(result, list) else [], payload.get('status', '')

records, logs = [], []
checkpoint_result = DATA_DIR / f'ars_stop_api_result_{DATE}_checkpoint.csv'
checkpoint_log = None  # query log 파일은 새로 만들지 않음
for i, row in targets.iterrows():
    params = {'apikey': API_KEY, 'sdCd': row['sdCd'], 'sttnArsno': row['sttnArsno']}
    if row.get('sggCd', ''): params['sggCd'] = row['sggCd']
    try:
        response = requests.get(ENDPOINT, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json(); items, status = parse(payload)
        for item in items:
            item = dict(item); item.update({'조회노선번호': row['노선번호'], '조회ARS': row['sttnArsno'], '조회시도코드': row['sdCd']}); records.append(item)
        logs.append({'노선번호': row['노선번호'], 'sttnArsno': row['sttnArsno'], 'sdCd': row['sdCd'], 'status': status, 'count': len(items), 'error': ''})
        print(f'[{i+1}/{len(targets)}] {row["노선번호"]} / {row["sttnArsno"]}: {len(items)}건')
    except Exception as exc:
        logs.append({'노선번호': row['노선번호'], 'sttnArsno': row['sttnArsno'], 'sdCd': row['sdCd'], 'status': 'ERROR', 'count': 0, 'error': repr(exc)})
        print(f'[{i+1}/{len(targets)}] {row["노선번호"]} / {row["sttnArsno"]}: 실패 - {exc}')
    # 매 요청 후 체크포인트 저장: 중간 중단 시 앞선 결과 보존
    pd.DataFrame(records).to_csv(checkpoint_result, index=False, encoding='utf-8-sig')
    time.sleep(0.1)

api_result = pd.DataFrame(records); query_log = pd.DataFrame(logs)
api_result.to_csv(DATA_DIR / f'ars_stop_api_result_{DATE}.csv', index=False, encoding='utf-8-sig')

# API 결과의 sttnId로 위경도 통합 파일의 누락된 정류장명을 보정
if not api_result.empty:
    api_result['sttnId'] = api_result['sttnId'].astype(str).str.strip()
    api_result = api_result.drop_duplicates('sttnId').set_index('sttnId')
    for date in [DATE]:
        path = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_coords.csv'
        out = pd.read_csv(path, dtype=str, encoding='utf-8-sig').fillna('')
        for sid, item in api_result.iterrows():
            name = item.get('sttnNm', '')
            ride = out['ride_sttn_id'].eq(sid)
            goff = out['goff_sttn_id'].eq(sid)
            out.loc[ride & out['승차정류장명'].eq(''), '승차정류장명'] = name
            out.loc[goff & out['하차정류장명'].eq(''), '하차정류장명'] = name
        output = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_coords_api_filled.csv'
        out.to_csv(output, index=False, encoding='utf-8-sig')
        print(f'보정 파일 저장: {output}')

## 2200번 ARS 31044부터 재조회

앞 셀 실행이 `31044`에서 중단된 경우 이 셀만 실행해 `31044` 이상 구간을 다시 조회하고, 기존 결과와 합칩니다.

### 셀 2. 2200번 조회 결과 확인\n

### ?? ? 2. 2200? ?? ???? ARS ?? BusStop API ??


In [ ]:
# 2200번 ARS 31044 이상 구간만 재조회
ars = pd.read_csv(ARS_INPUT, dtype=str, encoding='utf-8-sig').fillna('')
ars['sttnArsno'] = ars['sttnArsno'].str.strip().str.zfill(5)
tail = ars[(ars['노선번호'].eq('2200')) & (pd.to_numeric(ars['sttnArsno'], errors='coerce') >= 31044)].drop_duplicates(['sttnArsno','sdCd','sggCd']).reset_index(drop=True)
print(f'2200 재조회 대상: {len(tail)}개 (ARS 31044부터)')
tail_records, tail_logs = [], []
for i, row in tail.iterrows():
    params = {'apikey': API_KEY, 'sdCd': row['sdCd'], 'sttnArsno': row['sttnArsno']}
    if row.get('sggCd', ''): params['sggCd'] = row['sggCd']
    try:
        response = requests.get(ENDPOINT, params=params, timeout=30)
        response.raise_for_status(); payload = response.json(); items, status = parse(payload)
        for item in items:
            item = dict(item); item.update({'조회노선번호':'2200','조회ARS':row['sttnArsno'],'조회시도코드':row['sdCd']}); tail_records.append(item)
        tail_logs.append({'노선번호':'2200','sttnArsno':row['sttnArsno'],'sdCd':row['sdCd'],'status':status,'count':len(items),'error':''})
        print(f'[{i+1}/{len(tail)}] 2200 / {row["sttnArsno"]}: {len(items)}건')
    except Exception as exc:
        tail_logs.append({'노선번호':'2200','sttnArsno':row['sttnArsno'],'sdCd':row['sdCd'],'status':'ERROR','count':0,'error':repr(exc)})
        print(f'[{i+1}/{len(tail)}] 2200 / {row["sttnArsno"]}: 실패 - {exc}')
    time.sleep(0.1)

result_path = DATA_DIR / f'ars_stop_api_result_{DATE}.csv'
log_path = DATA_DIR / f'ars_stop_api_query_log_{DATE}.csv'
old_result = pd.read_csv(result_path, dtype=str, encoding='utf-8-sig').fillna('') if result_path.exists() else pd.DataFrame()
old_log = pd.read_csv(log_path, dtype=str, encoding='utf-8-sig').fillna('') if log_path.exists() else pd.DataFrame()
combined_result = pd.concat([old_result, pd.DataFrame(tail_records)], ignore_index=True).drop_duplicates(['sttnId','조회ARS'], keep='last') if tail_records or not old_result.empty else pd.DataFrame()
combined_log = pd.concat([old_log, pd.DataFrame(tail_logs)], ignore_index=True).drop_duplicates(['노선번호','sttnArsno','sdCd'], keep='last') if tail_logs or not old_log.empty else pd.DataFrame()
combined_result.to_csv(result_path, index=False, encoding='utf-8-sig')

# 새 API 결과로 위경도 통합 파일의 누락 정류장명 보정
if not combined_result.empty:
    for _, item in combined_result.iterrows():
        sid = str(item.get('sttnId','')).strip(); name = item.get('sttnNm','')
        if not sid or not name: continue
        path = DATA_DIR / f'gtx_a_transport_card_{DATE}_raw_with_coords.csv'
        out = pd.read_csv(path, dtype=str, encoding='utf-8-sig').fillna('')
        out.loc[out['ride_sttn_id'].eq(sid) & out['승차정류장명'].eq(''), '승차정류장명'] = name
        out.loc[out['goff_sttn_id'].eq(sid) & out['하차정류장명'].eq(''), '하차정류장명'] = name
        out.to_csv(DATA_DIR / f'gtx_a_transport_card_{DATE}_raw_with_coords_api_filled.csv', index=False, encoding='utf-8-sig')
print(f'2200 재조회 결과 합치기 완료: 성공 {len(tail_records)}건')